In [ ]:
import os
import sys
import json
import random
import shutil
import urllib.request
from pathlib import Path

import pandas as pd
import faker
import kagglehub
from dotenv import load_dotenv
from pymongo import MongoClient

# Raíz del proyecto (funciona como script y como notebook)
try:
    RAIZ = Path(__file__).resolve().parent.parent
except NameError:
    cwd = Path.cwd()
    RAIZ = cwd.parent if cwd.name == "src" else cwd

load_dotenv(RAIZ / ".env")
MONGO_URI = os.getenv("MONGO_URI", "mongodb://localhost:27017/")
MONGO_DB = os.getenv("MONGO_DB", "GameCon")


def conectar_mongo():
    """Usa el MongoDB real si responde; si no, cae a mongomock (en memoria)."""
    try:
        cliente = MongoClient(MONGO_URI, serverSelectionTimeoutMS=2000)
        cliente.admin.command("ping")
        print(f"Conectado a MongoDB: {MONGO_URI}")
    except Exception:
        import mongomock
        cliente = mongomock.MongoClient()
        print("MongoDB no disponible: usando mongomock (en memoria)")
    return cliente


client = conectar_mongo()
db = client[MONGO_DB]

## transformador.py

In [ ]:
RAW = RAIZ / "datos" / "raw"
RAW.mkdir(parents=True, exist_ok=True)


def descargar_kaggle():
    print("Descargando datasets de Kaggle...")
    ruta_vgsales = kagglehub.dataset_download("gregorut/videogamesales")
    ruta_anime2025 = kagglehub.dataset_download("rafidahmed816/anime-dataset-2025")

    for ruta in [ruta_vgsales, ruta_anime2025]:
        for root, _, files in os.walk(ruta):
            for f in files:
                if f.endswith(".complete"):
                    continue
                origen = os.path.join(root, f)
                destino = RAW / f
                if not destino.exists():
                    shutil.copy2(origen, destino)
                    print(f"Copiado a raw: {f}")
                else:
                    print(f"Ya existe en raw: {f}")

        # Elimina la carpeta de caché y su contenido
        shutil.rmtree(ruta, ignore_errors=True)
        print(f"Caché eliminada: {ruta}")

def descargar_goodreads():
    nombre="goodreads_books_comics_graphic.json.gz"
    destino = RAW/nombre

    if destino.exists():
        print(f"ya se encuentra {nombre} en la carpeta")
        return

    url=f"https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_books_comics_graphic.json.gz"
    with urllib.request.urlopen(url) as r, open(destino, "wb") as f:
        shutil.copyfileobj(r, f)



_necesarios = ["vgsales.csv", "anime-dataset-2025.csv", "goodreads_books_comics_graphic.json.gz"]
if all((RAW / f).exists() for f in _necesarios):
    print("Datasets ya descargados en datos/raw")
else:
    descargar_kaggle()
    descargar_goodreads()

## perfilar.py

In [ ]:
RAW = RAIZ / "datos" / "raw"

def cargar_datos_raw():
    """Carga y retorna los DataFrames crudos sin alterar sus índices."""
    df_games = pd.read_csv(RAW / "vgsales.csv")
    df_anime = pd.read_csv(RAW / "anime-dataset-2025.csv")
    df_comics = pd.read_json(
        RAW / "goodreads_books_comics_graphic.json.gz",
        lines=True,
        nrows=1000
    )
    return df_games, df_anime, df_comics

if True:
    df_games, df_anime, df_comics = cargar_datos_raw()
    print("Lectura cruda completada.")

## limpieza.py

In [ ]:
def obtener_datos_limpios():
    df_games, df_anime, df_comics = cargar_datos_raw()

    # 1. Juegos
    df_games = df_games.copy().drop_duplicates()
    cols_eliminar_games = ['Platform', 'NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']
    df_games = df_games.drop(columns=[c for c in cols_eliminar_games if c in df_games.columns])
    df_games = df_games.rename(columns={'Name': 'name', 'Publisher': 'productor', 'Genre': 'genero'})
    df_games = df_games.dropna(subset=['productor', 'name'])
    df_games['genero'] = df_games['genero'].astype(str).str.strip()
    df_games['dominio'] = 'juego'

    # 2. Anime
    df_anime = df_anime.copy().drop_duplicates()
    cols_eliminar_anime = ['Type', 'Source', 'Episodes', 'Status', 'Airing', 'Aired From', 'Aired To', 'Synopsis']
    df_anime = df_anime.drop(columns=[c for c in cols_eliminar_anime if c in df_anime.columns])
    df_anime = df_anime.rename(columns={'Title': 'name', 'Studios': 'productor', 'Genres': 'genero'})
    df_anime = df_anime.dropna(subset=['productor', 'name'])
    df_anime['dominio'] = 'anime'

    # 3. Cómics
    df_comics = df_comics.copy().drop_duplicates(subset=['book_id'])
    cols_a_borrar_comics = [
        'text_reviews_count', 'series', 'language_code', 'asin', 
        'is_ebook', 'kindle_asin', 'format', 'link', 'authors', 'num_pages', 
        'publication_day', 'isbn13', 'publication_month', 'edition_information', 
        'publication_year', 'url', 'image_url', 'work_id', 'title_without_series', 'country_code',
        'isbn', 'average_rating', 'description','similar_books', 'book_id', 'rating_count"'
    ]
    df_comics = df_comics.drop(columns=[c for c in cols_a_borrar_comics if c in df_comics.columns])
    df_comics = df_comics.rename(columns={'title': 'name', 'publisher': 'productor'})
    df_comics = df_comics.dropna(subset=['productor', 'name'])
    df_comics['genero'] = 'Comic/Manga'
    df_comics['dominio'] = 'manga_comic'


    return df_games, df_anime, df_comics


for _df in obtener_datos_limpios():
    print(_df.shape)
    display(_df.head(3))


def vacio_a_nulo(column):
    if column is None:
        return None

    limpio = column.astype("string").str.strip()
    basura = {"", "nan", "none", "misc", "null", "n/a", "-"}
    
    return limpio.mask(limpio.str.lower().isin(basura))




def quitar_tildes(x):
    if x != str(x):        
        return x
    x = x.replace("á", "a").replace("é", "e").replace("í", "i").replace("ó", "o").replace("ú", "u")
    return x

## procesar_tags.py

In [ ]:
df_games, df_anime, df_comics = obtener_datos_limpios()

def normalizar(tag):
    return str(tag).strip().lower().replace(" ", "_")

def generar_stands(df_games, df_anime, df_comics):
    stands = []

    # 1. Procesar Juegos (Publisher -> Stand)
    for productor, group in df_games.groupby('productor'):
        generos = [
            normalizar(g) for g in group['genero'].dropna().unique()
            if pd.notna(g) and str(g).strip().lower() not in ['misc', 'nan', '']
        ]
        stands.append({
            "nombre": str(productor),
            "dominio": "juego",
            "ubicacion": f"Zona Juegos - Stand {len(stands) + 1}",
            "tags": generos
        })

    # 2. Procesar Anime (Studios -> Stand)
    for productor, group in df_anime.groupby('productor'):
        raw_series = group['genero'].dropna().astype(str)
        raw_exploded = raw_series.str.split(',').explode().astype(str).str.strip()
        generos = [
            normalizar(g) for g in raw_exploded.unique()
            if g and g.lower() not in ['nan', 'none', 'misc', '']
        ]
        stands.append({
            "nombre": str(productor),
            "dominio": "anime",
            "ubicacion": f"Zona Anime - Stand {len(stands) + 1}",
            "tags": generos
        })

    # 3. Procesar Comics/Mangas (Publisher -> Stand)
    for productor, group in df_comics.groupby('productor'):
        stands.append({
            "nombre": str(productor),
            "dominio": "manga_comic",
            "ubicacion": f"Zona Comics - Stand {len(stands) + 1}",
            "tags": ["comic/manga"]
        })

    return pd.DataFrame(stands)

def generar_vocabulario_tags(df_stands):
    todos_los_tags = set(tag for tags_list in df_stands['tags'] for tag in tags_list)

    vocabulario = []
    for tag in todos_los_tags:
        frecuencia = sum(1 for tags_list in df_stands['tags'] if tag in tags_list)
        vocabulario.append({
            "_id": tag,
            "nivel": "general" if frecuencia > 10 else "especifico",
            "origenes": [tag],
            "frecuencia": frecuencia
        })
    return pd.DataFrame(vocabulario)




if True:
    df_stands = generar_stands(df_games, df_anime, df_comics)
    df_tags = generar_vocabulario_tags(df_stands)
    df_stands.dropna(how='all')


    print(f"Stands generados: {len(df_stands)}")
    print(df_stands)

    print(f"Tags en vocabulario: {len(df_tags)}")
    df_tags = df_tags.sort_values(by='frecuencia', ascending=False)
    print(df_tags)

## clases.py

In [ ]:
from typing import List, Dict, Optional
from bson import ObjectId

class Tag: 
    """Vocabulario controlado y tabla de mapeo (D2)"""
    def __init__(self, id_tag: str, nivel: str, origenes: List[str], frecuencia: int = 0):
        self._id = id_tag  # String normalizado
        self.nivel = nivel  # general o específico
        self.origenes = origenes  # Géneros originales mapeados
        self.frecuencia = frecuencia

    def to_dict(self) -> dict:
        return {
            "_id": self._id,
            "nivel": self.nivel,
            "origenes": self.origenes,
            "frecuencia": self.frecuencia
        }

class Stand: 
    """Representación de un Productor/Stand (D1)"""
    def __init__(self, nombre: str, dominio: str, ubicacion: str, tags: List[str], _id: Optional[ObjectId] = None):
        self._id = _id or ObjectId()
        self.nombre = nombre
        self.dominio = dominio  # juego, anime, manga_comic, musica
        self.ubicacion = ubicacion
        self.tags = tags

    def to_dict(self) -> dict:
        return {
            "_id": self._id,
            "nombre": self.nombre,
            "dominio": self.dominio,
            "ubicacion": self.ubicacion,
            "tags": self.tags
        }

class Presentacion: 
    """Eventos del escenario"""
    def __init__(self, titulo: str, inicio: str, fin: str, tags: List[str], stand_id: ObjectId, _id: Optional[ObjectId] = None):
        self._id = _id or ObjectId()
        self.titulo = titulo
        self.inicio = inicio
        self.fin = fin
        self.tags = tags
        self.stand_id = stand_id

    def to_dict(self) -> dict:
        return {
            "_id": self._id,
            "titulo": self.titulo,
            "inicio": self.inicio,
            "fin": self.fin,
            "tags": self.tags,
            "stand_id": self.stand_id
        }

class Asistente: 
    """Perfil del asistente (D3, D4)"""
    def __init__(self, nombre: str, intereses_declarados: List[str], intereses_deducidos: Optional[Dict[str, int]] = None, _id: Optional[ObjectId] = None):
        self._id = _id or ObjectId()
        self.nombre = nombre
        self.intereses_declarados: List[str] = intereses_declarados[:8]  # Tope de 8 intereses declarados
        self.intereses_deducidos = intereses_deducidos or {}

    def to_dict(self) -> dict:
        return {
            "_id": self._id,
            "nombre": self.nombre,
            "intereses_declarados": self.intereses_declarados,
            "intereses_deducidos": self.intereses_deducidos
        }

class Interaccion: 
    """Registro de eventos/interacciones (D3)"""
    def __init__(self, asistente_id: ObjectId, objetivo_id: ObjectId, tipo: str, fecha: str, _id: Optional[ObjectId] = None):
        self._id = _id or ObjectId()
        self.asistente_id = asistente_id
        self.objetivo_id = objetivo_id
        self.tipo = tipo  # visita, asistencia, favorito
        self.fecha = fecha

    def to_dict(self) -> dict:
        return {
            "_id": self._id,
            "asistente_id": self.asistente_id,
            "objetivo_id": self.objetivo_id,
            "tipo": self.tipo,
            "fecha": self.fecha
        }

## inicio_mongodb.py

In [ ]:
def inicializar_colecciones():
    colecciones = ["tags", "stands", "presentaciones", "asistentes", "interacciones"]
    for col in colecciones:
        if col not in db.list_collection_names():
            db.create_collection(col)
            print(f"Colección '{col}' creada.")
        else:
            print(f"Colección '{col}' ya existe.")

inicializar_colecciones()

## cargar_mongo.py

In [ ]:
def cargar_datos_a_mongo():
    print(f"Conectando a MongoDB ({MONGO_DB})...")

    # 1. Obtener DataFrames limpios y procesados
    print("Generando stands y vocabulario desde los datasets...")
    df_games, df_anime, df_comics = obtener_datos_limpios()
    df_stands = generar_stands(df_games, df_anime, df_comics)
    df_tags = generar_vocabulario_tags(df_stands)

    # 2. Cargar stands (idempotente: vaciamos y volvemos a insertar)
    print(f"Cargando {len(df_stands)} stands...")
    db.stands.delete_many({})
    docs_stands = df_stands.to_dict(orient="records")
    db.stands.insert_many(docs_stands)
    print(f"-> {db.stands.count_documents({})} stands en MongoDB.")

    # 3. Cargar tags
    print(f"Cargando {len(df_tags)} tags...")
    db.tags.delete_many({})
    docs_tags = df_tags.to_dict(orient="records")
    db.tags.insert_many(docs_tags)
    print(f"-> {db.tags.count_documents({})} tags en MongoDB.")

    print("\n¡Carga completada con éxito!")


cargar_datos_a_mongo()

## crear_asistentes2.py

In [ ]:
from pathlib import Path
import pandas as pd
import random
import faker

#configuración inicial
fake = faker.Faker('es_CL')  #genera nombres 
NUM_USUARIOS = 100     #asistentes a simular

#vocabulario controlado, se puede añadir mas si es necesario 
tags_validos = [
    "action", "adventure", "fantasy", "sci-fi", "horror", "mystery",
    "romance", "slice_of_life", "sports", "supernatural", "suspense",
    "isekai", "mecha", "psychological", "strategy", "role-playing",
    "shooter", "racing", "comedy", "drama"
]

datos = []
for i in range(1, NUM_USUARIOS + 1):
    #id s
    asistente_id = f"ASIS_{i:04d}"
    
    nombre = fake.name()
    
    #3 gustos unidos por coma para q se vea bien en un CSV
    gustos_aleatorios = random.sample(tags_validos, 3)
    gustos_declarados = ", ".join(gustos_aleatorios)
    
    datos.append({
        "asistente_id": asistente_id,
        "nombre": nombre,
        "gustos_declarados": gustos_declarados
    })

#guardar como CSV
df = pd.DataFrame(datos)

ruta_salida = RAIZ / "asistentes_simulados.csv"
df.to_csv(ruta_salida, index=False, encoding="utf-8")

print(f"¡Éxito! Se generaron {NUM_USUARIOS} asistentes en: {ruta_salida}")
print("\nVista previa de los datos:")
print(df.head())

# Cargar los asistentes simulados a MongoDB
docs_asistentes = [
    {
        "_id": fila.asistente_id,
        "nombre": fila.nombre,
        "intereses_declarados": [g.strip() for g in fila.gustos_declarados.split(",")],
        "intereses_deducidos": {},
    }
    for fila in df.itertuples()
]
db.asistentes.delete_many({})
db.asistentes.insert_many(docs_asistentes)
print(f"-> {db.asistentes.count_documents({})} asistentes en MongoDB.")

## perfil_deducido.py

In [ ]:
def registrar_interaccion(asistente_id, objetivo_id, tipo_interaccion, tags_objetivo):
    """Registra una interacción y actualiza el perfil deducido del asistente."""
    #guardar la interaccion (sin tope)
    db.interacciones.insert_one({
        "asistente_id": asistente_id,
        "objetivo_id": objetivo_id,
        "tipo": tipo_interaccion
    })

    #actualizar el perfil deducido
    peso = 2 if tipo_interaccion == "favorito" else 1
    for tag in tags_objetivo:
        db.asistentes.update_one(
            {"_id": asistente_id},
            {"$inc": {f"intereses_deducidos.{tag}": peso}}
        )
    print(f"Interacción '{tipo_interaccion}' registrada, perfil actualizado")

## recomendaciones.py

In [ ]:
def normalizar_id(asistente_id: str) -> str:
    limpio = asistente_id.strip()
    if limpio.isdigit():
        return f"ASIS_{int(limpio):04d}"
    if limpio.lower().startswith("asis_"):
        num_part = limpio[5:]
        if num_part.isdigit():
            return f"ASIS_{int(num_part):04d}"
    return limpio.upper()


def obtener_perfil_asistente(asistente_id: str):
    id_busqueda = normalizar_id(asistente_id)
    asistente = db.asistentes.find_one({"_id": id_busqueda})
    if not asistente:
        asistente = db.asistentes.find_one({"_id": asistente_id.strip()})

    if not asistente:
        return None, []

    declarados = asistente.get("intereses_declarados", []) or []
    deducidos = list(asistente.get("intereses_deducidos", {}).keys()) if asistente.get("intereses_deducidos") else []
    gustos_totales = sorted(list(set(declarados + deducidos)))
    return asistente, gustos_totales


def recomendar_stands(asistente_id: str, top_n: int = 5):
    asistente, gustos_usuario = obtener_perfil_asistente(asistente_id)
    if not asistente:
        return None

    if not gustos_usuario:
        return []

    pipeline = [
        {"$match": {"tags": {"$in": gustos_usuario}}},
        {"$project": {
            "nombre": 1,
            "dominio": 1,
            "ubicacion": 1,
            "tags": 1,
            "tags_coincidentes": {"$setIntersection": ["$tags", gustos_usuario]},
            "coincidencias": {"$size": {"$setIntersection": ["$tags", gustos_usuario]}}
        }},
        {"$sort": {"coincidencias": -1, "nombre": 1}},
        {"$limit": top_n}
    ]
    try:
        return list(db.stands.aggregate(pipeline))
    except NotImplementedError:
        # mongomock no implementa $setIntersection: se calcula en Python
        gustos = set(gustos_usuario)
        resultado = []
        for st in db.stands.find({"tags": {"$in": gustos_usuario}}):
            comunes = [t for t in st["tags"] if t in gustos]
            st["tags_coincidentes"] = comunes
            st["coincidencias"] = len(comunes)
            resultado.append(st)
        resultado.sort(key=lambda x: (-x["coincidencias"], x["nombre"]))
        return resultado[:top_n]


def modo_interactivo():
    print("-" * 50)
    print("Recomendador de Stands - GameCon")
    print("Escriba 'salir' o 'q' para terminar.")
    print("-" * 50)

    while True:
        try:
            entrada = input("\nID del asistente: ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\nFin del programa.")
            break

        if not entrada or entrada.lower() in ("salir", "exit", "q"):
            print("Fin del programa.")
            break

        asistente, gustos = obtener_perfil_asistente(entrada)
        if not asistente:
            print(f"Error: Asistente '{entrada}' no encontrado.")
            continue

        declarados = asistente.get("intereses_declarados", []) or []
        deducidos = list(asistente.get("intereses_deducidos", {}).keys()) if asistente.get("intereses_deducidos") else []

        print(f"\nAsistente: {asistente.get('nombre', 'Sin nombre')} ({asistente['_id']})")
        print(f"Tags declarados: {', '.join(declarados) if declarados else 'Ninguno'}")
        if deducidos:
            print(f"Tags deducidos: {', '.join(deducidos)}")
        print(f"Tags a coincidir: {', '.join(gustos) if gustos else 'Sin tags'}")

        if not gustos:
            print("Aviso: El asistente no tiene tags asociados.")
            continue

        try:
            top_str = input("Cantidad de recomendaciones [5]: ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\nFin del programa.")
            break

        top_n = int(top_str) if top_str.isdigit() and int(top_str) > 0 else 5
        recomendaciones = recomendar_stands(asistente["_id"], top_n=top_n)

        if not recomendaciones:
            print("No se encontraron stands con coincidencias.")
            continue

        print(f"\nTop {len(recomendaciones)} Stands:")
        for i, stand in enumerate(recomendaciones, 1):
            coincidencias = stand.get("coincidencias", 0)
            tags_matched = ", ".join(stand.get("tags_coincidentes", []))
            ubicacion = stand.get("ubicacion", "Sin ubicacion")
            dominio = stand.get("dominio", "General")

            print(f"  {i}. {stand['nombre']} [{dominio}] ({ubicacion})")
            print(f"     Coincidencias ({coincidencias}): {tags_matched}")


# Demo no interactiva (para el modo interactivo: modo_interactivo())
registrar_interaccion("ASIS_0001", "demo", "favorito", ["action", "fantasy"])
for i, st in enumerate(recomendar_stands("ASIS_0001", top_n=5), 1):
    print(f"{i}. {st['nombre']} [{st['dominio']}] -> {st['tags_coincidentes']}")